<a href="https://colab.research.google.com/github/GabzBarbosa/Resumo-de-avalia-es/blob/main/avaliacoes_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
!pip install gspread google-auth pandas numpy matplotlib fpdf


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import auth
import gspread
from google.auth import default
from fpdf import FPDF


In [50]:
# ===== CONFIGURAÇÕES =====
SPREADSHEET_ID = "1zEn8jrIBOIQwkHmQQ880v8SUpFO5sONHFQ7LWpwH9PM"
SHEET_NAME = "base"

# 'semanal' ou 'mensal'
MODO = "mensal"

# =========================


In [51]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


In [52]:
worksheet = gc.open_by_key(SPREADSHEET_ID).worksheet(SHEET_NAME)
df = pd.DataFrame(worksheet.get_all_records())


In [53]:
df["nota"] = pd.to_numeric(df["Nota"], errors="coerce")
df["avaliacao"] = df["Avaliação"].fillna("").str.lower()

df = df.dropna(subset=["nota"])


In [54]:
if MODO == "mensal":
    PERCENTIL_VOLUME = 0.75
else:
    PERCENTIL_VOLUME = 0.70


In [55]:
def agregar_entidade(df, coluna):
    agg = (
        df.groupby(coluna)
        .agg(
            total_avaliacoes=("nota", "count"),
            media_nota=("nota", "mean")
        )
        .reset_index()
    )
    agg["score_ponderado"] = agg["media_nota"] * agg["total_avaliacoes"]
    limite_volume = agg["total_avaliacoes"].quantile(PERCENTIL_VOLUME)
    return agg, limite_volume


In [56]:
def gerar_rankings(agg, limite):
    melhores = (
        agg[agg["total_avaliacoes"] >= limite]
        .sort_values(
            by=["total_avaliacoes", "media_nota"],
            ascending=[False, False]
        )
        .head(10)
    )

    piores = (
        agg[
            (agg["media_nota"] < 3) &
            (agg["total_avaliacoes"] >= limite)
        ]
        .sort_values(
            by=["total_avaliacoes", "media_nota"],
            ascending=[False, True]
        )
        .head(10)
    )
    return melhores, piores


In [57]:
PALAVRAS_NEGATIVAS = [
    "ruim", "péssimo", "horrível", "defeito",
    "troca", "devolução", "não recomendo",
    "quebrado", "problema", "demora"
]

def resumo_sentimento(df, coluna, valor):
    textos = df[df[coluna] == valor]["avaliacao"]
    achados = [p for p in PALAVRAS_NEGATIVAS if textos.str.contains(p).any()]
    return ", ".join(achados) if achados else "Sem padrão claro"


In [59]:
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# =====================
# TÍTULO
# =====================
pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, "Dashboard Executivo - Avaliações de Clientes", ln=True)

pdf.set_font("Arial", size=10)
pdf.multi_cell(
    0, 6,
    f"Relatório gerado no modo: {MODO.upper()}.\n"
    "Critério de análise: entidades mais avaliadas, "
    "com priorização de impacto (volume) e qualidade (nota média)."
)

# =====================
# FUNÇÃO AUXILIAR – TABELA SIMPLES
# =====================
def escrever_tabela(pdf, df, coluna_nome):
    pdf.set_font("Arial", "B", 10)
    pdf.cell(70, 8, coluna_nome, border=1)
    pdf.cell(30, 8, "Volume", border=1)
    pdf.cell(30, 8, "Média", border=1)
    pdf.cell(30, 8, "Score", border=1)
    pdf.ln()

    pdf.set_font("Arial", size=9)
    for _, row in df.iterrows():
        pdf.cell(70, 8, str(row[coluna_nome])[:40], border=1)
        pdf.cell(30, 8, str(row["total_avaliacoes"]), border=1)
        pdf.cell(30, 8, f"{row['media_nota']:.2f}", border=1)
        pdf.cell(30, 8, f"{row['score_ponderado']:.1f}", border=1)
        pdf.ln()

# =====================
# SELLERS / MARCAS / PRODUTOS
# =====================
for entidade, coluna in {
    "Seller": "Seller",
    "Marca": "Marca",
    "Produto": "Produto"
}.items():

    pdf.add_page()
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 10, f"{entidade} - Top Avaliados", ln=True)

    escrever_tabela(pdf, resultados[entidade]["melhores"], coluna)

    pdf.ln(5)
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 10, f"{entidade} - Piores Avaliados", ln=True)

    escrever_tabela(pdf, resultados[entidade]["piores"], coluna)

    # =====================
    # RESUMO DESCRITIVO (SENTIMENTO)
    # =====================
    pdf.ln(3)
    pdf.set_font("Arial", "B", 11)
    pdf.cell(0, 8, "Principais motivos de insatisfação:", ln=True)

    pdf.set_font("Arial", size=9)
    for _, row in resultados[entidade]["piores"].iterrows():
        pdf.multi_cell(
            0, 6,
            f"- {row[coluna]}: {row.get('Resumo Sentimento', 'Sem padrão claro')}"
        )

# =====================
# GRÁFICOS EXECUTIVOS
# =====================
pdf.add_page()
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 10, "Visão Executiva - Gráficos", ln=True)

for img in [
    "distribuicao_notas.png",
    "top_sellers.png",
    "piores_sellers.png"
]:
    pdf.ln(5)
    pdf.image(img, w=180)

# =====================
# SALVAR PDF
# =====================
nome_pdf = f"dashboard_avaliacoes_{MODO}.pdf"
pdf.output(nome_pdf)

nome_pdf


'dashboard_avaliacoes_mensal.pdf'

In [60]:
print(resultados["Seller"]["piores"].columns)


Index(['Seller', 'total_avaliacoes', 'media_nota', 'score_ponderado',
       'Resumo Sentimento'],
      dtype='object')
